### Automated Milling Workflow

In [ ]:
%load_ext autoreload
%autoreload 2

from odemis.acq.millmng import mill_patterns, MillingTaskSettings, MillingSettings2, MillingPatternParameters
from odemis.acq.feature import CryoFeature
from odemis.util.filename import make_unique_name
from pprint import pprint
from odemis import model
from odemis.acq.acqmng import acquire

from odemis.acq.stream import LiveStream, SEMStream, FIBStream, StaticSEMStream
import os

from datetime import datetime 
timestamp = datetime.timestamp(datetime.now())
# format as YYYY-MM-DD_HH-MM-SS
timestamp = datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d_%H-%M-%S')
PROJECT_PATH = f"/home/patrick/development/scratch/project/automated-milling/{timestamp}"
os.makedirs(PROJECT_PATH, exist_ok=True)


In [ ]:
# connect to hardware

# stage
stage = model.getComponent(role="stage-bare")
print(f"Stage Position: {stage.position.value}")

# setup electron beam, det
electron_beam = model.getComponent(role="e-beam")
electron_det = model.getComponent(role="se-detector")
electron_focus = model.getComponent(role="ebeam-focus")

hwemtvas = set()
hwdetvas = set()

hwemt_vanames = ("beamCurrent", "accelVoltage", "resolution", "dwellTime", "horizontalFoV")
hwdet_vanames = ("brightness", "contrast", "detector_mode", "detector_type")
for vaname in model.getVAs(electron_beam):
    if vaname in hwemt_vanames:
        hwemtvas.add(vaname)
for vaname in model.getVAs(electron_det):
    if vaname in hwdet_vanames:
        hwdetvas.add(vaname)

sem_stream = SEMStream(
    name="SEM",
    detector=electron_det,
    dataflow=electron_det.data,
    emitter=electron_beam,
    focuser=electron_focus,
    hwemtvas=hwemtvas,
    hwdetvas=hwdetvas,
    blanker=None)

# setup ion beam, det
ion_beam = model.getComponent(role="ion-beam")
ion_det = model.getComponent(role="se-detector-ion")
ion_focus = model.getComponent(role="ion-focus")

hwemtvas = set()
hwdetvas = set()
# hwemt_vanames = ("beamCurrent", "accelVoltage", "resolution", "dwellTime", "horizontalFoV")
# hwdet_vanames = ("brightness", "contrast", "detector_mode", "detector_type")
for vaname in model.getVAs(ion_beam):
    if vaname in hwemt_vanames:
        hwemtvas.add(vaname)
for vaname in model.getVAs(ion_det):
    if vaname in hwdet_vanames:
        hwdetvas.add(vaname)

fib_stream = FIBStream(
    name="FIB",
    detector=ion_det,
    dataflow=ion_det.data,
    emitter=ion_beam,
    focuser=ion_focus,
    hwemtvas=hwemtvas,
    hwdetvas=hwdetvas,
)

In [ ]:
f = acquire([sem_stream, fib_stream])

data, err  = f.result()

sem_image, fib_image = data
%matplotlib inline
from matplotlib import pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(sem_image, cmap="gray")
ax[1].imshow(fib_image, cmap="gray")
plt.show()

In [ ]:

class CryoLamellaFeature(object):
    def __init__(self, name: str, position: dict, 
                 milling_tasks: dict, 
                 status: str,
                 alignment: dict = None): 
        self.name = model.StringVA(name)
        self.position = model.VigilantAttribute(position, unit=["m", "m", "m", "rad", "rad"])                
        self.milling_tasks: dict = milling_tasks
        self.alignment: dict = alignment
        self.status: str = model.StringVA(status)
        self.reference_image = None
        self.project = None
        self.project_path = None
        self.path = None

    def to_json(self):
        return {
            "name": self.name.value,
            "position": self.position.value,
            "milling_tasks": {k: v.to_json() for k, v in self.milling_tasks.items()},
            "alignment": self.alignment,
            "status": self.status.value,
            "project": self.project,
            "project_path": self.project_path,
            "path": self.path,
        }
    
    @classmethod
    def from_json(cls, data: dict):
        feature = cls(
            name=data["name"],
            position=data["position"],
            milling_tasks={k: MillingTaskSettings.from_json(v) for k, v in data["milling_tasks"].items()},
            alignment=data["alignment"],
            status=data["status"]
        )
        feature.project = data["project"]
        feature.project_path = data["project_path"]
        feature.path = data["path"]
        
        return feature

    def __repr__(self):
        return f"CryoLamellaFeature({self.name}, {self.position}, {self.milling_tasks}, {self.alignment}, {self.status})"

# save a list of positions (features)
features = []

# load milling settings
MILLING_START, MILLING_ROUGH, MILLING_FINE, MILL_POLISHING = "Start Milling", "Rough Milling", "Fine Milling", "Polishing" 

milling_tasks = {
    MILLING_ROUGH: {},
    MILLING_FINE: {},
    MILL_POLISHING: {},
}

task_dict = {
    "milling": {
        "current": 0.2e-9,
        "voltage": 30e3,
        "field_of_view": 80e-6,
        "mode": "Serial", 
    },
    "patterns": [
        {"name": "Rectangle 1",
         "width": 10e-6,
          "height": 6e-6,
          "depth": 4e-6,
          "rotation": 0,
          "center_x": 0,
          "center_y": 5e-6,
          "scan_direction": "TopToBottom"}, 
        {"name": "Rectangle 2",
         "width": 10e-6,
          "height": 6e-6,
          "depth": 4e-6,
          "rotation": 0,
          "center_x": 0,
          "center_y": -5e-6,
          "scan_direction": "TopToBottom"}, 
    ]
}

milling_tasks[MILLING_ROUGH] = MillingTaskSettings.from_json(task_dict)
milling_tasks[MILLING_FINE] = MillingTaskSettings.from_json(task_dict)
milling_tasks[MILL_POLISHING] = MillingTaskSettings.from_json(task_dict)

# features
feature = CryoLamellaFeature(
    "Lamella-01",
    {"x": 50e-6, "y": 25e-6, "z": 15e-6, "rx": 0, "rz": 0},
    milling_tasks=milling_tasks,
    status=MILLING_START,
)
feature.reference_image = fib_image
features.append(feature)

# features
feature = CryoLamellaFeature(
    "Lamella-02",
    {"x": 33e-6, "y": 100e-6, "z": 66e-6, "rx": 0, "rz": 0},
    milling_tasks=milling_tasks,
    status=MILLING_START,
)
feature.reference_image = fib_image
features.append(feature)

for f in features:
    print(f)
    f.project_path = PROJECT_PATH
    f.path = os.path.join(PROJECT_PATH, f.name.value)
    os.makedirs(f.path, exist_ok=True)

    print(f.path)
    

# steps
# 1. create feature
# 2. select position
# 3. select milling tasks
# 4. select alignment
# ready to mill

# TODO: ref image data -> when to acquire reference image
# TODO: save directory -> project directory / feature directory



In [ ]:

f = stage.moveAbs({"x": 0, "y": 0, "z": 0, "rx": 0, "rz": 0})
f.result()


In [ ]:
# loop through each position, mill lamella
from odemis.acq.align.shift import MeasureShift
from odemis.dataio import find_fittest_converter

# status
task_list = [MILLING_ROUGH, MILLING_FINE, MILL_POLISHING]
task_status: dict = {MILLING_ROUGH: "Rough Milling", 
                     MILLING_FINE: "Fine Milling", 
                     MILL_POLISHING: "Polishing"}

_exporter = find_fittest_converter("filename.ome.tiff")


for task_name in task_list:
    print(f"Starting {task_name} for {len(features)} features...")
    for feature in features:

        print(f"Feature: {feature.name.value}")
        print(f"Position: {feature.position.value}")
        print(f"Previous Status: {feature.status.value}")
        print("--" * 10)
        print(f"Starting {task_name} for {feature.name.value}")

        # move to position
        f = stage.moveAbs(feature.position.value)
        f.result()

        # beam shift alignment
        f = acquire([fib_stream])
        data, _ = f.result()
        new_image = data[0]

        ref_image = feature.reference_image
        shift_px = MeasureShift(ref_image, new_image, 10)
        shift_px = (1, 1)
        pixelsize = ref_image.metadata[model.MD_PIXEL_SIZE]
        shift_m = (shift_px[0] * pixelsize[0], shift_px[1] * pixelsize[1])

        previous_shift = ion_beam.shift.value
        print(f"Previous: {previous_shift}, Shift: {shift_m}")
        shift = (shift_m[0] + previous_shift[0], shift_m[1] + previous_shift[1])  # m
        ion_beam.shift.value = shift
        print(f"Shift: {ion_beam.shift.value}")

        # mill patterns
        # f = mill_patterns(feature.milling_tasks[task_name])
        # f.result()

        # acquire images
        f = acquire([sem_stream, fib_stream])
        data, ex = f.result()
        sem_image, fib_image = data

        # save images
        feature_name = feature.name.value
        sem_filename = os.path.join(feature.path, f"{feature_name}-{task_name}-Finished-SEM.ome.tiff").replace(" ", "-") # TODO: make unique 
        fib_filename = os.path.join(feature.path, f"{feature_name}-{task_name}-Finished-FIB.ome.tiff").replace(" ", "-") # TODO: make unique
        _exporter.export(sem_filename, sem_image)
        _exporter.export(fib_filename, fib_image)       
        break

        # plot images
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(sem_image, cmap="gray")
        ax[1].imshow(fib_image, cmap="gray")
        plt.show()

        # update status
        feature.status.value = task_status[task_name]

        print(f"Finished {task_name} for {feature.name.value}")

        # save feature

    print("-"*50)


In [ ]:
for f in features:
    pprint(f.to_json())    



In [ ]:
# save features to yaml

import yaml
filename = os.path.join(PROJECT_PATH, "features.yaml")
with open(filename, "w") as f:
    yaml.dump([f.to_json() for f in features], f)

In [ ]:
# load features from yaml
features = []
with open(filename, "r") as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
    for d in data:
        features.append(CryoLamellaFeature.from_json(d))

In [ ]:
for f in features:
    pprint(f.to_json())